In [ ]:
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_ollama import ChatOllama

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore


# =========================================================
# 1. CONTEXT
# =========================================================
# Context اطلاعات مربوط به اجرای فعلی Agent است.
#
# این اطلاعات قرار نیست توسط Agent به عنوان Memory ذخیره شوند.
# ما آنها را هنگام اجرای Agent به آن می‌دهیم.
# =========================================================

@dataclass
class AgentContext:
    user_id: str
    branch_code: str
    role: str


# =========================================================
# 2. STATE / SHORT-TERM MEMORY
# =========================================================
#
# create_agent خودش State مربوط به messages را مدیریت می‌کند.
#
# ما فعلاً State سفارشی اضافه نمی‌کنیم.
#
# messages همان conversation فعلی است.
#
# checkpointer باعث می‌شود این State بر اساس thread_id
# بین چند invoke حفظ شود.
# =========================================================


# =========================================================
# 3. LONG-TERM MEMORY
# =========================================================
#
# Store اطلاعاتی را نگه می‌دارد که باید بین conversationهای
# مختلف باقی بمانند.
# =========================================================

store = InMemoryStore()


# =========================================================
# 4. SHORT-TERM MEMORY
# =========================================================
#
# Checkpointer وضعیت هر Thread را نگه می‌دارد.
# =========================================================

checkpointer = InMemorySaver()


# =========================================================
# 5. MODEL
# =========================================================

model = ChatOllama(
    model="qwen3:1.7b",
    temperature=0,
)


# =========================================================
# 6. TOOL
# =========================================================
#
# این Tool هم Context را می‌بیند
# و هم Long-term Memory را.
#
# ToolRuntime به صورت خودکار توسط LangChain تزریق می‌شود
# و مدل آن را نمی‌بیند.
# =========================================================

@tool
def get_user_preferences(
    runtime: ToolRuntime[AgentContext],
) -> str:
    """
    Retrieve the user's long-term preferences.
    """

    # -----------------------------------------
    # خواندن Context
    # -----------------------------------------

    user_id = runtime.context.user_id

    # -----------------------------------------
    # خواندن Long-term Memory
    # -----------------------------------------

    memory = runtime.store.get(
        ("users",),
        user_id,
    )

    if memory is None:
        return "No saved preferences found."

    return str(memory.value)


# =========================================================
# 7. TOOL برای ذخیره ترجیح کاربر
# =========================================================

@tool
def save_user_preference(
    preference: str,
    runtime: ToolRuntime[AgentContext],
) -> str:
    """
    Save a user preference in long-term memory.
    """

    user_id = runtime.context.user_id

    # -----------------------------------------
    # ذخیره در Long-term Memory
    # -----------------------------------------

    runtime.store.put(
        ("users",),
        user_id,
        {
            "preference": preference,
        },
    )

    return "User preference saved successfully."


# =========================================================
# 8. CREATE AGENT
# =========================================================

agent = create_agent(
    model=model,

    tools=[
        get_user_preferences,
        save_user_preference,
    ],

    # Short-term Memory
    checkpointer=checkpointer,

    # Long-term Memory
    store=store,

    # Context
    context_schema=AgentContext,
)


# =========================================================
# 9. CONTEXT کاربر
# =========================================================

context = AgentContext(
    user_id="U1001",
    branch_code="101",
    role="BranchManager",
)


# =========================================================
# 10. THREAD
# =========================================================
#
# این Thread مربوط به یک conversation است.
# =========================================================

thread_1 = {
    "configurable": {
        "thread_id": "conversation-001"
    }
}


# =========================================================
# 11. پیام اول
# =========================================================

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "من گزارش‌ها را خلاصه و به صورت جدولی می‌خواهم."
                    " این ترجیح من را ذخیره کن."
                ),
            }
        ]
    },

    thread_1,

    context=context,
)


print(result["messages"][-1].content)


# =========================================================
# 12. پیام دوم
# =========================================================
#
# همان Thread
#
# بنابراین Agent پیام قبلی را به خاطر دارد.
# =========================================================

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "برای شعبه من یک گزارش آماده کن."
                ),
            }
        ]
    },

    thread_1,

    context=context,
)


print(result["messages"][-1].content)


# =========================================================
# 13. یک Thread جدید
# =========================================================
#
# conversation-002 یک conversation کاملاً جدید است.
#
# Short-term Memory قبلی را ندارد.
#
# اما Long-term Memory کاربر هنوز وجود دارد.
# =========================================================

thread_2 = {
    "configurable": {
        "thread_id": "conversation-002"
    }
}


result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "من معمولاً گزارش‌ها را چگونه می‌خواهم؟"
                ),
            }
        ]
    },

    thread_2,

    context=context,
)


print(result["messages"][-1].content)